# 23 · MCP tools with a local server

MCP exposes tools through a protocol rather than a Python import. The bundled server uses stdio and serves synthetic product data.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Optional: about 2 Lite calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


The VM's base ADK install does not include `mcp`. If needed, run `%pip install 'mcp>=1.8,<2'` once in this existing kernel; no environment activation is needed. The cell below reports the missing dependency without silently installing packages.


In [ ]:
import importlib.util
HAS_MCP = importlib.util.find_spec("mcp") is not None
if not HAS_MCP:
    print("Optional dependency missing: run %pip install 'mcp>=1.8,<2', then rerun this notebook.")
else:
    from mcp import StdioServerParameters
    from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
    from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
    toolset = McpToolset(connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(command=sys.executable, args=[str(ROOT / "mcp_server.py")]),
        timeout=20), tool_filter=["lookup_product"])
    try:
        lab = await make_lab(llm("mcp_shop", "Use lookup_product to check stock and price.", tools=[toolset]))
        await ask(lab, "What is the stock and price of P101?")
    finally:
        await toolset.close()


## Try it

Expose a second read-only tool from the MCP server. Tool filters limit what this agent can discover.


## Reference

[Official ADK documentation](https://adk.dev/tools-custom/mcp-tools/). Shared configuration: `config.json`. No environment activation or login cells are needed.
